<a href="https://colab.research.google.com/github/ozodbekAI/Data-Scince-and-AI-Portfolio/blob/main/Email_Spam_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("ashfakyeafi/spam-email-classification")

print("Path to dataset files:", path)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv(path + "/email.csv")
df.head()

In [ ]:
df.describe()

In [ ]:
df.duplicated().sum()

In [ ]:
df.drop_duplicates(inplace=True)

In [ ]:
df.duplicated().sum()

In [ ]:
df.drop(index=5572, inplace=True)

In [ ]:
df["Message Lenght"] = df["Message"].apply(len)

In [ ]:
df.head()

In [ ]:
df["Message"][1]

In [ ]:
sns.histplot(df["Message Lenght"], bins = 20)
plt.title("Distribution of Message Length")
plt.xlabel("Character")
plt.show()

In [ ]:
sns.boxplot(
    data=df,
    x = "Category",
    y = "Message Lenght"
)
plt.title("Distribution of Message Length by Category")
plt.show()

In [ ]:
df.groupby("Category")["Message Lenght"].mean()

In [ ]:
df.groupby("Category")["Message Lenght"].mean().plot(
    kind="bar",
    color=["steelblue", "tomato"]
)
plt.title("Average Message Length by Category")
plt.xlabel("Category")
plt.ylabel("Average Message Length")
plt.show()

In [ ]:
df.head()

In [ ]:
df["Word_Count"] = df["Message"].apply(lambda x: len(x.split()))

In [ ]:
df.head()

In [ ]:
df.groupby("Category")["Word_Count"].mean().plot(
    kind="bar",
    color=["steelblue", "tomato"]
)
plt.title("Average Word Count by Category")
plt.xlabel("Category")
plt.ylabel("Average Word Count")
plt.show()


In [ ]:
sns.histplot(
    data = df,
    x = "Word_Count",
    hue = "Category",
    bins = 40
)

In [ ]:
df["Category"] = df["Category"].map(
  {
      "spam": 1,
      "ham": 0
  }
)
df.head()

In [ ]:
import warnings
warnings.filterwarnings("ignore")


In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X = df["Message"]
y = df["Category"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(stop_words="english", max_features=5000, ngram_range=(1,2))
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.model_selection import RandomizedSearchCV

In [ ]:
models = {
    "Naive Bayes": (
        MultinomialNB(),
        {
            "alpha": [0.1, 0.5, 1.0, 2, 5]
        }
    ),
    "Logistic Regression": (
        LogisticRegression(max_iter=1000, class_weight="balanced"),
        {
            "C": [0.001, 0.01, 0.1, 1, 10, 100, 1000]
        }
    ),
    "Liner SVM": (
        LinearSVC(
            class_weight="balanced"
        ),
        {
            "C": [0.01, 0.1, 1, 10, 100]
        }
    ),
    "Random Forest": (
        RandomForestClassifier(class_weight="balanced"),
        {
            "n_estimators": [100, 200, 300, 400],
            "max_depth": [10, 20, 30, None],
            "min_samples_split": [2, 5, 10],
            "min_samples_leaf": [1, 2, 4]
        }
    ),

    "Decision Tree": (
        DecisionTreeClassifier(class_weight="balanced"),
        {
            "max_depth": [10, 20, 30, None],
            "min_samples_split": [2, 5, 10],
            "min_samples_leaf": [1, 2, 4]
        }
    ),

    "Gradient Boosting": (
        GradientBoostingClassifier(),
        {
            "n_estimators": [100, 200, 300, 400],
            "learning_rate": [0.01, 0.1, 0.5, 1],
            "max_depth": [1, 3, 5, 7],
        }
    ),

    "K-Nearest Neighbors": (
        KNeighborsClassifier(),
        {
            "n_neighbors": [3, 5, 7, 9]
        }
    )
}

In [ ]:
results = []

best_models = {}

for name, (model, params) in models.items():
  search = RandomizedSearchCV(
      estimator=model,
      param_distributions=params,
      n_iter=10,
      scoring="f1",
      n_jobs=-1,
      cv=5,
      verbose=1
  )

  search.fit(X_train_tfidf, y_train)

  best_model = search.best_estimator_

  best_models[name] = best_model

  predictions = best_model.predict(X_test_tfidf)

  accuracy = accuracy_score(y_test, predictions)
  precision = precision_score(y_test, predictions)
  recall = recall_score(y_test, predictions)
  f1 = f1_score(y_test, predictions)


  try:
    auc = roc_auc_score(y_test, best_model.decision_function(X_test_tfidf))
  except:
    try:
      auc = roc_auc_score(y_test, best_model.predict_proba(X_test_tfidf)[:, 1])
    except:
      auc = np.nan

  results.append({
      "Model": name,
      "Accuracy": accuracy,
      "Precision": precision,
      "Recall": recall,
      "F1 Score": f1,
      "AUC": auc,
      "Best Params": search.best_params_
  })

In [ ]:
 results

In [ ]:
results_df = pd.DataFrame(results)
results_df

In [ ]:
results_df.sort_values(
    by = ["F1 Score", "AUC"],
    ascending=False,
    inplace=True
)

In [ ]:
results_df

In [ ]:
results_df.style.background_gradient(
    cmap="Blues",
    subset=["Accuracy", "Precision", "Recall", "F1 Score", "AUC"]

)
#

In [ ]:
best_model_name = results_df.iloc[0]["Model"]
best_model = best_models[best_model_name]
best_model_name

In [ ]:
best_model

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

ConfusionMatrixDisplay.from_estimator(
    best_model,
    X_test_tfidf,
    y_test,
    cmap="Blues",
)
plt.show()


In [ ]:
emails = [
    "Congratulations! You won a free iPhone. Click here now.",
    "Hey, are we still meeting tomorrow at 10 AM?",
    "URGENT! Claim your 1000$ Walmart gift card today"
]

email_vectorized = tfidf.transform(emails)

predictions = best_model.predict(email_vectorized)

for email, prediction in zip(emails, predictions):
    print(f"Email: {email}")
    print(f"Prediction: {'Spam' if prediction == 1 else 'Ham'}")
    print()

In [ ]:
import pickle

with open("spam_classifier.pkl", "wb") as f:
    pickle.dump(best_model, f)

with open("tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(tfidf, f)